# Семинар: Chat Completion API, Reasoning, Grounding и Tool Calling


---

На этом семинаре мы работаем с моделью **Qwen3.5-35B-A3B-FP8**, развёрнутой через **vLLM**. Все запросы выполняются через стандартный OpenAI-совместимый Chat Completion API с помощью библиотеки `requests` — никаких специфичных клиентов.

**Что изучим:**
1. Базовый Chat Completion API — структура запроса и ответа
2. Reasoning (мышление вслух) — как включить и что получить
3. Мультимодальность — изображения по URL и в base64
4. Structured output — JSON-схема для ответов
5. Grounding — задачи детекции и поиска ключевых точек
6. Streaming — поточная генерация токенов
7. Sampling — влияние параметров на поведение модели
8. Token log-probabilities — распределение вероятностей токенов
9. Tool Calling — создание инструментов и их вызов моделью

In [ ]:
import requests
import json
import base64
import re
import os
import math
from urllib.request import urlretrieve
from IPython.display import display, Image, Markdown
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# ─── Точки входа ────────────────────────────────────────────────────────────
MODEL_URL  = os.getenv("MODEL_URL")
MODEL_NAME = "Qwen/Qwen3.5-397B-A17B-FP8"

HEADERS = {"Content-Type": "application/json"}

# Вспомогательная функция: отправить запрос и вернуть полный JSON
def chat(messages, **kwargs):
    if 'chat_template_kwargs' not in kwargs:
        kwargs['chat_template_kwargs'] = {"enable_thinking": False}
        
    payload = {"model": MODEL_NAME, "messages": messages, **kwargs}
    
    resp = requests.post(MODEL_URL, headers=HEADERS, json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()

# Извлечь текст первого choice из ответа
def get_text(response):
    return response["choices"][0]["message"]["content"]

# Извлечь reasoning-блок (thinking) если он есть
def get_reasoning(response):
    msg = response["choices"][0]["message"]
    return msg.get("reasoning", None)

print("Конфигурация загружена. MODEL_NAME =", MODEL_NAME, " MODEL_URL is None =", (MODEL_URL is None))

---
## Часть 1. Базовый запрос и структура ответа

### Chat Completion API: структура сообщений

Стандартный OpenAI-совместимый API принимает список `messages`, каждое из которых — словарь `{role, content}`. Роли:

| Роль | Смысл |
|------|-------|
| `system` | Системный промпт — «личность» и контекст модели |
| `user` | Сообщение пользователя |
| `assistant` | Предыдущие ответы модели (для multi-turn) |
| `tool` | Результат вызова инструмента (рассмотрим в части 9) |

Перед тем как попасть в трансформер, список сообщений преобразуется в плоскую строку с помощью **chat template** — Jinja2-шаблона, специфичного для каждой архитектуры. Для Qwen3 это выглядит примерно так:

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Привет!<|im_end|>
<|im_start|>assistant
<think>
... (reasoning block) ...
</think>
Привет! Чем могу помочь?<|im_end|>
```

Спецтокены `<|im_start|>` / `<|im_end|>` обозначают границы реплик. Блок `<think>...</think>` — внутреннее рассуждение, которое **не входит** в `content`, но доступно в поле `reasoning_content` ответа (при включённом reasoning).

> **Интересный момент:** Chat template хранится прямо в токенайзере модели — в `tokenizer_config.json`, поле `chat_template`. Это позволяет воспроизводить один и тот же препроцессинг локально через `tokenizer.apply_chat_template(messages)`.

In [ ]:
messages = [
    {"role": "system", "content": "Ты полезный ассистент."},
    {"role": "user",   "content": "Объясни в двух предложениях, что такое свёрточная нейронная сеть."}
]

response = chat(messages, temperature=0.7, max_tokens=256)

print("=== Полный JSON ответа ===")
print(json.dumps(response, ensure_ascii=False, indent=2))

### Анатомия ответа

Ключевые поля JSON-ответа:

```
{
  "id": "chatcmpl-...",          // уникальный ID запроса
  "model": "Qwen/Qwen3.5-...",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "...",          // итоговый ответ
        "reasoning_content": null  // null если reasoning выключен
      },
      "finish_reason": "stop"      // stop | length | tool_calls
    }
  ],
  "usage": {
    "prompt_tokens": 42,
    "completion_tokens": 128,
    "total_tokens": 170
  }
}
```

`finish_reason`:
- `stop` — модель закончила ответ сама (встретила `<|im_end|>`)
- `length` — достигнут лимит `max_tokens`
- `tool_calls` — модель решила вызвать инструмент

In [ ]:
print("=== Текст ответа ===")
print(get_text(response))

print("\n=== Использование токенов ===")
usage = response["usage"]
print(f"  prompt:     {usage['prompt_tokens']}")
print(f"  completion: {usage['completion_tokens']}")
print(f"  total:      {usage['total_tokens']}")

---
## Часть 2. Reasoning — «думать вслух»

### Что такое reasoning-модели?

Reasoning-модели обучаются с помощью **RLVR** (Reinforcement Learning with Verifiable Rewards): модель генерирует внутренний «черновик» рассуждения (Chain-of-Thought), а затем финальный ответ. Награда начисляется за **верность финального ответа**, но не за качество промежуточных шагов — это позволяет модели самостоятельно вырабатывать стратегии мышления.

В Qwen3 рассуждение обрамлено тегами `<think>...</think>`. vLLM возвращает его в отдельном поле `reasoning_content` (не смешивая с `content`), что удобно для парсинга.

**Включение reasoning** в API: параметр `"enable_thinking": true` (передаётся в `chat_template_kwargs` или как vLLM-специфичный флаг). При выключенном режиме модель всё равно может «думать» — но блок `<think>` будет пустым или отсутствовать.

> **Важно:** токены внутри `<think>` **тарифицируются**, но не видны в `content`. Следите за `completion_tokens` — при сложных задачах reasoning может занимать сотни токенов.

In [ ]:
MATH_PROBLEM = """
Нужно решить задачку по математике:
Сумма всех различных натуральных делителей некоторого натурального числа на 6 больше, чем само это число. Найдите это число. Если ответов несколько, укажите их все через пробел в порядке возрастания.

Внимательно подумай, попробуй разные подходы к решению. Проверь свое решение.
Докажи, что твой ответ верен. Будь более-менее кратким и ответь в <5 предложений.
"""

messages_math = [
    {"role": "user", "content": MATH_PROBLEM}
]


In [ ]:

# Запрос БЕЗ reasoning
resp_no_think = chat(
    messages_math,
    temperature=0.7,
    max_tokens=32000,
    chat_template_kwargs={"enable_thinking": False}
)


# Запрос С reasoning
resp_think = chat(
    messages_math,
    temperature=0.7,
    max_tokens=32000,
    chat_template_kwargs={"enable_thinking": True}
)

In [ ]:
print("=== Без reasoning ===")
print(get_text(resp_no_think))
print(f"\nТокенов: {resp_no_think['usage']['completion_tokens']}")

print("=== С reasoning ===")
print(get_text(resp_think))
print(f"\nТокенов: {resp_think['usage']['completion_tokens']}")

In [ ]:
reasoning = get_reasoning(resp_think)
answer    = get_text(resp_think)

print("=== Reasoning-блок (первые 40000 символов) ===")
if reasoning:
    print(reasoning[:4000], "...")
else:
    # Попытка вытащить <think>...</think> из content вручную
    raw = answer
    m = re.search(r"<think>(.*?)</think>", raw, re.DOTALL)
    if m:
        print(m.group(1)[:4000], "...")
    else:
        print("[reasoning_content отсутствует]")

print("\n=== Финальный ответ ===")
print(answer)

print(f"\nТокенов (с thinking): {resp_think['usage']['completion_tokens']}")
print(f"Токенов (без thinking): {resp_no_think['usage']['completion_tokens']}")

---
## Часть 3. Мультимодальность — отправка изображений

### Как VLM принимает картинки?

В мультимодальных моделях (например Qwen-VL) изображение прежде всего прогоняется через **Vision Encoder** (обычно ViT), который разбивает его на патчи и возвращает последовательность визуальных эмбеддингов. Затем они вставляются в текстовую последовательность через специальные **placeholder-токены** (`<|vision_start|>...<|vision_end|>`).

Chat Completion API поддерживает два способа передать изображение:

1. **По URL** — модель (или vLLM-сервер) скачивает картинку сама:
   ```python
   {"type": "image_url", "image_url": {"url": "https://..."}}
   ```

2. **Base64** — изображение кодируется и передаётся прямо в теле запроса (не нужен доступ к интернету от сервера):
   ```python
   {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64,<BASE64>"}}
   ```

Поле `content` в `user`-сообщении при мультимодальном запросе — **список** объектов, а не строка.

Изображения для этого семинара — панорамы Санкт-Петербурга:

In [ ]:
IMG_URL_1 = "https://upload.wikimedia.org/wikipedia/commons/thumb/0/0c/Spb_Vasilievsky_Island_Neva_at_SchmidtEmb_asv2019-09_crop.jpg/1920px-Spb_Vasilievsky_Island_Neva_at_SchmidtEmb_asv2019-09_crop.jpg"
IMG_URL_2 = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/23/Spb_06-2017_img03_Spit_of_Vasilievsky_Island.jpg/1920px-Spb_06-2017_img03_Spit_of_Vasilievsky_Island.jpg"

# Отобразим изображения
display(Image(url=IMG_URL_1, width=700))
display(Image(url=IMG_URL_2, width=700))

In [ ]:
# --- Способ 1: передача по URL ---
# У меня не отработает, нет сетевого доступа

# messages_img_url = [
#     {
#         "role": "user",
#         "content": [
#             {"type": "image_url", "image_url": {"url": IMG_URL_1}},
#             {"type": "text", "text": "Что изображено на этой фотографии? Опиши кратко архитектурные объекты."}
#         ]
#     }
# ]

# resp_url = chat(messages_img_url, max_tokens=512)
# print("=== Ответ (изображение по URL) ===")
# print(get_text(resp_url))

In [ ]:
# --- Способ 2: передача как base64 ---

# Скачаем изображение и закодируем
img_bytes_1 = requests.get(IMG_URL_1, timeout=30, headers={'user-agent': "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}).content
img_b64_1   = base64.b64encode(img_bytes_1).decode("utf-8")
data_uri_1 = f"data:image/jpeg;base64,{img_b64_1}"

print(f"Размер base64-строки: {len(img_b64_1):,} символов")

messages_img_b64 = [
    {
        "role": "user",
        "content": [
            {"type": "image_url", 
             "image_url": {"url": data_uri_1}
            },
            {"type": "text", "text": "Что изображено на этой фотографии? Опиши кратко архитектурные объекты."}
        ]
    }
]

resp_b64 = chat(messages_img_b64, max_tokens=512)
print("\n=== Ответ (base64) ===")
print(get_text(resp_b64))


> **Интересный момент:** base64-кодирование увеличивает размер данных примерно в 4/3 раза (каждые 3 байта → 4 ASCII-символа). Для изображения 1920px это легко несколько мегабайт в теле HTTP-запроса — учитывайте при проектировании pipeline.

---
## Часть 4. Работа с несколькими изображениями

Несколько изображений передаются простым добавлением объектов `image_url` в список `content`. Модель видит их в том порядке, в каком они перечислены, что важно для задач сравнения или анализа последовательностей.

In [ ]:
img_bytes_2 = requests.get(IMG_URL_2, timeout=30, headers={'user-agent': "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}).content
img_b64_2   = base64.b64encode(img_bytes_2).decode("utf-8")
data_uri_2 = f"data:image/jpeg;base64,{img_b64_2}"


messages_two_imgs = [
    {
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": data_uri_1}},
            {"type": "image_url", "image_url": {"url": data_uri_2}},
            {
                "type": "text",
                "text": (
                    "Перед тобой две фотографии одного города. "
                    "Назови 3 общие черты и 3 ключевых отличия между снимками. "
                    "Отвечай структурированно."
                )
            }
        ]
    }
]

resp_two = chat(messages_two_imgs, max_tokens=768)
print(get_text(resp_two))

---
## Часть 5. Structured Output — JSON-ответы по схеме

### Зачем нужен structured output?

В production-системах ответ модели часто нужно дальше парсить программно. Просьба «отвечай в JSON» не гарантирует корректность — модель может добавить пояснение или нарушить структуру. **Structured output** использует **constrained decoding**: на каждом шаге маски запрещают токены, несовместимые с ожидаемой JSON-схемой. Это гарантирует валидность синтаксиса.

В vLLM / OpenAI API это параметр `response_format`:

```python
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "my_schema",
        "schema": { ... }  # JSON Schema draft-7
    }
}
```

> **Под капотом:** vLLM использует библиотеку [outlines](https://github.com/outlines-dev/outlines) для построения конечного автомата (FSM) из JSON Schema. На каждом шаге декодирования допустимые следующие токены вычисляются как пересечение словаря модели с допустимыми переходами FSM.

In [ ]:
comparison_schema = {
    "type": "object",
    "properties": {
        "city": {"type": "string"},
        "common_features": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 3,
            "maxItems": 3
        },
        "differences": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "aspect": {"type": "string"},
                    "image_1": {"type": "string"},
                    "image_2": {"type": "string"}
                },
                "required": ["aspect", "image_1", "image_2"]
            },
            "minItems": 3,
            "maxItems": 3
        }
    },
    "required": ["city", "common_features", "differences"]
}

messages_structured = [
    {
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": data_uri_1}},
            {"type": "image_url", "image_url": {"url": data_uri_2}},
            {
                "type": "text",
                "text": "Сравни два изображения. Назови 3 общие черты и 3 отличия."
            }
        ]
    }
]

resp_structured = chat(
    messages_structured,
    max_tokens=1024,
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "image_comparison",
            "schema": comparison_schema
        }
    }
)

raw_json = get_text(resp_structured)
parsed   = json.loads(raw_json)

print(json.dumps(parsed, indent=4, ensure_ascii=False))

---
## Часть 6. Grounding — детекция объектов и ключевых точек

### Как Qwen-VL реализует grounding?

**Grounding** — это способность модели не только описать, но и **локализовать** объект на изображении, выдав координаты.

Qwen3-VL использует специальные токены для описания пространственных объектов:

```
<|object_ref_start|>кот<|object_ref_end|>
<|box_start|>(x1,y1),(x2,y2)<|box_end|>
```

Координаты нормализованы к диапазону `[0, 1000]` относительно размера входного изображения (не пикселей!). Например, `(250,300),(750,800)` означает прямоугольник от 25% до 75% ширины и от 30% до 80% высоты.

Для **ключевых точек** (keypoints) используется схожая нотация:
```
<|point_start|>(x,y)<|point_end|>
```

**Как модель этому обучилась?**  
В обучающем датасете Qwen-VL есть пары (изображение, JSON с аннотациями bounding boxes). При обучении задача формулируется как языковая: «Найди объект X» → модель генерирует координаты в виде текста. Таким образом, пространственное понимание «встроено» в языковую голову без отдельной детекционной ветви.

In [ ]:
# Попросим модель найти конкретный объект на панораме
messages_grounding = [
    {
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": data_uri_2}},
            {
                "type": "text",
                "text": (
                    "Please detect the Rostral Column (Ростральная колонна) in this image. "
                    "Return bounding box coordinates in json format: "
                )
            }
        ]
    }
]

resp_grounding = chat(messages_grounding, max_tokens=256, chat_template_kwargs={"enable_thinking": False})
raw_ground = get_text(resp_grounding)
print("Ответ модели:")
print(raw_ground)

In [ ]:
import imageio.v3 as imageio
# Парсим bounding box из ответа модели
def parse_bbox(text):
    """Извлекает (x1,y1,x2,y2) нормализованные координаты [0..1000] из ответа Qwen-VL."""
    pattern = r"\[(\d+), (\d+), (\d+), (\d+)\]"
    m = re.search(pattern, text)
    if not m:
        return None
    return tuple(int(v) for v in m.groups())

bbox = parse_bbox(raw_ground)
print("Распознанный bbox (нормализованный):", bbox)

if bbox:
    x1, y1, x2, y2 = bbox
    # Загружаем изображение для визуализации
    from PIL import Image as PILImage
    import io

    img_bytes2 = requests.get(IMG_URL_2, timeout=30).content

    fig, ax = plt.subplots(1, 1, figsize=(14, 7))
    ax.imshow(imageio.imread(img_bytes_2))

    rect = patches.Rectangle(
        (x1, y1),
        (x2 - x1),
        (y2 - y1),
        linewidth=3, edgecolor='red', facecolor='none'
    )
    ax.add_patch(rect)
    # ax.text(x1, y1, "Rostral Column", color='red', fontsize=12, weight='bold')
    ax.set_axis_off()
    plt.title("Grounding: Qwen3-VL bounding box")
    plt.tight_layout()
    plt.show()
else:
    print("Bounding box не найден в ответе — возможно, модель вернула координаты в другом формате.")
    print("Полный ответ:", raw_ground)

---
## Часть 7. Streaming — поточная генерация

### Как работает streaming?

При `stream=True` сервер не ждёт завершения генерации и отправляет **Server-Sent Events (SSE)** — по одному дельта-объекту на каждый сгенерированный токен (или небольшую пачку):

```
data: {"choices":[{"delta":{"content":"Привет"}, "finish_reason":null}]}
data: {"choices":[{"delta":{"content":"!"}, "finish_reason":null}]}
data: {"choices":[{"delta":{}, "finish_reason":"stop"}]}
data: [DONE]
```

Каждая строка — JSON с полем `delta.content` (или `delta.reasoning_content` для reasoning-токенов). Ключевые параметры:
- `stream=True` — включить стриминг
- `stream_options={"include_usage": true}` — финальный чанк будет содержать `usage`

> При reasoning-стриминге сначала идут чанки с `delta.reasoning_content`, затем — `delta.content`. Это отражает реальный порядок генерации.

In [ ]:
SHOW_RAW_CHUNKS = False  # Поставьте True, чтобы видеть сырые SSE-строки

payload_stream = {
    "model": MODEL_NAME,
    "messages": messages_math,
    "max_tokens": 2048,
    "temperature": 0.7,
    "stream": True,
    "stream_options": {"include_usage": True},
    "chat_template_kwargs": {"enable_thinking": False}
}

thinking_buf = []
answer_buf   = []
usage_info   = None

with requests.post(MODEL_URL, headers=HEADERS, json=payload_stream,
                   stream=True, timeout=120) as r:
    r.raise_for_status()
    for raw_line in r.iter_lines():
        if not raw_line:
            continue
        line = raw_line.decode("utf-8")

        if SHOW_RAW_CHUNKS:
            print("RAW:", line)

        if line.startswith("data: "):
            data_str = line[6:]
            if data_str.strip() == "[DONE]":
                break
            chunk = json.loads(data_str)

            # Забираем usage из последнего чанка
            if chunk.get("usage"):
                usage_info = chunk["usage"]

            delta = chunk["choices"][0]["delta"] if chunk.get("choices") else {}

            if delta.get("reasoning_content"):
                thinking_buf.append(delta["reasoning_content"])
                print("\x1b[90m" + delta["reasoning_content"] + "\x1b[0m", end="", flush=True)
            elif delta.get("content"):
                answer_buf.append(delta["content"])
                print(delta["content"], end="", flush=True)

print("\n\n=== Итог ===")
print(f"Reasoning-токенов: ~{len(''.join(thinking_buf).split())}  слов")
print(f"Answer-токенов:    ~{len(''.join(answer_buf).split())}  слов")
if usage_info:
    print(f"Usage: {usage_info}")

---
## Часть 8. Параметры сэмплирования

### Математика сэмплирования

На каждом шаге декодирования трансформер выдаёт логиты $ z_i \in \mathbb{R}^{|V|} $ для каждого токена словаря $ V $. Вероятность токена $ i $:

$$ p_i = \text{softmax}(z / T)_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}} $$

где **температура** $ T > 0 $:
- $ T \to 0 $: greedy decoding — выбирается токен с наибольшим логитом
- $ T = 1 $: оригинальное распределение модели
- $ T > 1 $: «размытие» распределения, повышение энтропии

Дополнительные стратегии фильтрации:

| Параметр | Смысл |
|----------|-------|
| `top_p` | Nucleus sampling: оставляем наименьший префикс токенов, суммарная вероятность которых ≥ p |
| `top_k` | Оставляем только k токенов с наибольшей вероятностью |
| `repetition_penalty` | Штраф за повторение уже использованных токенов: $ z_i \leftarrow z_i / r $ если токен $ i $ уже встречался |
| `min_p` | Отсекаем токены с $ p_i < \text{min\_p} \cdot \max_j p_j $ |

**Порядок применения** в vLLM: `top_k` → `top_p` → `min_p` → `temperature` → `repetition_penalty` → сэмплирование.

In [ ]:
sampling_configs = [
    {"label": "T=0 (greedy)",     "temperature": 0.0,  "top_p": 1.0},
    {"label": "T=0.3 (focused)",  "temperature": 0.3,  "top_p": 0.9},
    {"label": "T=1.0 (default)",  "temperature": 1.0,  "top_p": 1.0},
    {"label": "T=1.5 (creative)", "temperature": 1.5,  "top_p": 1.0},
    {"label": "T=10 (chaos)",     "temperature": 10.0, "top_p": 1.0},
]

print("Задача:", MATH_PROBLEM.strip())
print("=" * 60)

for cfg in sampling_configs:
    resp = chat(
        messages_math,
        temperature=cfg["temperature"],
        top_p=cfg["top_p"],
        max_tokens=1024,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}}
    )
    text = get_text(resp)
    print(f"\n[{cfg['label']}]")
    print(text)

---
## Часть 9. Log-probabilities — распределение вероятностей токенов

### Зачем смотреть на log-probs?

Поле `logprobs` в ответе позволяет получить **вероятности альтернативных токенов** на каждом шаге генерации. Это полезно для:
- Диагностики уверенности модели
- Реализации собственных стратегий декодирования
- Оценки перплексии
- Отладки structured output

Параметр `logprobs=True` + `top_logprobs=N` возвращает `N` лучших токенов с log-вероятностями на каждом шаге:

```json
"logprobs": {
  "content": [
    {
      "token": "Привет",
      "logprob": -0.023,
      "top_logprobs": [
        {"token": "Привет",  "logprob": -0.023},
        {"token": "Здравст", "logprob": -4.1},
        ...
      ]
    },
    ...
  ]
}
```

In [ ]:
logprob_configs = [
    {"label": "T=0.2",  "temperature": 0.2},
    {"label": "T=1.0",  "temperature": 1.0},
    {"label": "T=2.0",  "temperature": 2.0},
]

logprob_results = {}

for cfg in logprob_configs:
    resp = chat(
        messages_math,
        temperature=cfg["temperature"],
        max_tokens=512,
        logprobs=True,
        top_logprobs=5,
        chat_template_kwargs={"enable_thinking": False}
    )
    print(get_text(resp)[:128] + '...')
    logprob_results[cfg["label"]] = resp["choices"][0].get("logprobs", {}).get("content", [])

print("Данные получены. Строим визуализацию...")

In [ ]:
# logprob_results['T=1.0'][:5]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (label, lp_data) in zip(axes, logprob_results.items()):
    scores = []
    for tk in lp_data:
        scores.append(math.exp(tk['logprob']))

    ax.hist(scores, bins=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 20)
    ax.grid()

    ax.set_title(label)
    ax.set_xlabel('Proba')

plt.tight_layout()
plt.show()

**Интерпретация:** При низкой температуре (T=0.3) один токен доминирует с высокой вероятностью — модель «уверена». При T=2.0 вероятности выравниваются — энтропия растёт, ответы становятся случайнее. На графике видно, как температура «перераспределяет» массу вероятности.

---
## Часть 10. Tool Calling — вызов инструментов

### Как устроен tool calling?

Tool calling — механизм, при котором модель **не выполняет** действие сама, а возвращает структурированный запрос на вызов внешней функции. Схема взаимодействия:

```
User → [messages + tools] → Model → tool_calls JSON
                                          ↓
                                   Ваш код исполняет функцию
                                          ↓
User → [messages + tool result] → Model → Финальный ответ
```

Инструменты описываются как JSON Schema в параметре `tools`:

```json
{
  "type": "function",
  "function": {
    "name": "execute_python",
    "description": "Executes Python code and returns stdout.",
    "parameters": {
      "type": "object",
      "properties": {
        "code": {"type": "string", "description": "Python code to execute"}
      },
      "required": ["code"]
    }
  }
}
```

Когда модель решает вызвать инструмент, `finish_reason` = `"tool_calls"`, а в `message` появляется поле `tool_calls` вместо `content`.

### MCP — Model Context Protocol

MCP (Anthropic, 2024) — открытый стандарт для описания и вызова инструментов через унифицированный протокол. По сути это HTTP/stdio-транспорт поверх JSON-RPC, где сервер экспонирует набор **tools**, **resources** и **prompts**. Идея аналогична LSP (Language Server Protocol) в IDE: любой клиент может подключиться к любому MCP-серверу. В данном семинаре мы реализуем логику вручную, но принципы те же.

In [ ]:
import subprocess
import sys

# Инструмент: выполнить Python-код в subprocess и вернуть stdout
def execute_python_tool(code: str) -> str:
    try:
        result = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True, text=True, timeout=15
        )
        stdout = result.stdout.strip()
        stderr = result.stderr.strip()
        output = stdout if stdout else ""
        if stderr:
            output += ("\n[stderr] " + stderr) if output else ("[stderr] " + stderr)
        return output or "(no output)"
    except subprocess.TimeoutExpired:
        return "[error] Timeout exceeded"
    except Exception as e:
        return f"[error] {e}"


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "execute_python",
            "description": (
                "Execute arbitrary Python 3 code in a subprocess and return its stdout. "
                "Use this to perform calculations, network requests, inspect system state, etc."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "Valid Python 3 code to execute"
                    }
                },
                "required": ["code"]
            }
        }
    }
]

print("Инструмент execute_python готов к использованию.")

In [ ]:
# Диспетчер вызовов инструментов
def dispatch_tool(name: str, arguments: dict) -> str:
    if name == "execute_python":
        return execute_python_tool(arguments["code"])
    return f"[error] Unknown tool: {name}"


# Агентный цикл: модель вызывает инструменты пока не решит задачу
def run_agent(system_prompt: str, user_message: str, max_steps: int = 6):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message}
    ]

    for step in range(max_steps):
        print(f"\n--- Шаг {step + 1} ---")
        resp = chat(
            messages,
            tools=TOOLS,
            tool_choice="auto",
            max_tokens=1024,
            temperature=0.3,
            chat_template_kwargs={"enable_thinking": False}
        )

        choice      = resp["choices"][0]
        finish      = choice["finish_reason"]
        msg         = choice["message"]

        # Добавляем ответ модели в историю
        messages.append(msg)

        if finish == "tool_calls":
            for tc in msg.get("tool_calls", []):
                fn_name = tc["function"]["name"]
                fn_args = json.loads(tc["function"]["arguments"])
                call_id = tc["id"]

                print(f"  [TOOL CALL] {fn_name}")
                print(f"  Код:\n{fn_args.get('code', '')}")

                tool_result = dispatch_tool(fn_name, fn_args)
                print(f"  Результат: {tool_result[:300]}")

                # Добавляем результат как tool-сообщение
                messages.append({
                    "role": "tool",
                    "tool_call_id": call_id,
                    "content": tool_result
                })

        elif finish == "stop":
            print("\n=== Финальный ответ ===")
            print(msg.get("content", ""))
            return messages

    print("[warn] Достигнут лимит шагов")
    return messages

In [ ]:
SYSTEM_TOOL = """
You are a helpful AI assistant with access to a Python execution environment.
When you need to check system information, run calculations, or make network requests — use the execute_python tool.
Think step by step. Always explain what you are going to do before calling a tool.
If a network request fails, think about what environment variables or system settings might help.
"""

USER_TASK = """
Determine the IP address of the host where you are currently running.
Try multiple methods: socket, HTTP requests to external services, etc.
Report all results you find.
"""

history = run_agent(SYSTEM_TOOL, USER_TASK)

In [ ]:
# Подсказываем модели про переменную окружения HTTP_PROXY
# (Если сетевые запросы не прошли на предыдущем шаге)

HINT_MESSAGE = """
Note: this environment may have an HTTP_PROXY environment variable set that could be useful 
for making outbound network requests. Try reading it and using it in your requests.
"""

# Продолжаем разговор с подсказкой
history.append({"role": "user", "content": HINT_MESSAGE})

for step in range(4):
    resp = chat(
        history,
        tools=TOOLS,
        tool_choice="auto",
        max_tokens=1024,
        temperature=0.3,
        chat_template_kwargs={"enable_thinking": False}
    )

    choice = resp["choices"][0]
    finish = choice["finish_reason"]
    msg    = choice["message"]
    history.append(msg)

    print(f"--- Шаг (с подсказкой) {step+1} ---")

    if finish == "tool_calls":
        for tc in msg.get("tool_calls", []):
            fn_name = tc["function"]["name"]
            fn_args = json.loads(tc["function"]["arguments"])
            call_id = tc["id"]

            print(f"  [TOOL] {fn_name}:\n{fn_args.get('code', '')}")
            result = dispatch_tool(fn_name, fn_args)
            print(f"  => {result[:400]}")

            history.append({
                "role": "tool",
                "tool_call_id": call_id,
                "content": result
            })

    elif finish == "stop":
        print("\n=== Финальный ответ ===")
        print(msg.get("content", ""))
        break

### Итоги семинара

| Тема | Ключевые параметры API |
|------|------------------------|
| Базовый запрос | `messages`, `max_tokens`, `temperature` |
| Reasoning | `chat_template_kwargs: {enable_thinking: true}`, поле `reasoning_content` |
| Изображения | `content: [{type: image_url}, {type: text}]` |
| Structured output | `response_format: {type: json_schema, json_schema: ...}` |
| Grounding | Специальные токены `<\|box_start\|>`, нормализация [0, 1000] |
| Streaming | `stream: true`, `stream_options: {include_usage: true}` |
| Сэмплирование | `temperature`, `top_p`, `top_k`, `repetition_penalty` |
| Log-probs | `logprobs: true`, `top_logprobs: N` |
| Tool calling | `tools`, `tool_choice`, `finish_reason: tool_calls` |

---
*Семинар по курсу «Компьютерное зрение» — ФКН ВШЭ / МФТИ*